<a href="https://colab.research.google.com/github/faizanahmed-ai/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/faizanahmed-ai/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### My Rule

The baseline rule identifies **high-ranking pages that are attracting fewer clicks than expected**. For each page, I compare its actual CTR with the average CTR of other pages in the same search position bucket (1 to 3, 4 to 10, 11 to 20, or 21+). The difference becomes the **CTR gap**, which serves as the page's priority score. A larger positive gap indicates a stronger opportunity to improve click-through rate by rewriting the page title. Pages with no positive gap receive no action.

### Reason Codes

* **CTR_UNDERPERFORM**: The page's CTR is below the expected CTR for its ranking position, indicating it is underperforming relative to similar-ranked pages and is a candidate for a title rewrite.
* **NO_ACTION**: The page's CTR meets or exceeds the expected value for its position, so no title optimization is recommended.


In [16]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

df['position_bucket'] = pd.cut(df['avg_position'], bins=[0,3,10,20,1000], labels=['1-3','4-10','11-20','21+'])
signal1 = df.groupby('position_bucket', observed=True).agg(avg_ctr=('ctr','mean'), n=('ctr','count')).reset_index()

df['staleness_bucket'] = pd.cut(df['days_since_last_update'], bins=[0,30,90,180,10000], labels=['0-30','31-90','91-180','180+'])
signal2 = df.groupby('staleness_bucket', observed=True).agg(avg_ctr=('ctr','mean'), n=('ctr','count')).reset_index()

print(signal1)
print(signal2)

  position_bucket   avg_ctr      n
0             1-3  2.714303   1141
1            4-10  0.651045  11842
2           11-20  0.323443   7273
3             21+  0.211333   8539
  staleness_bucket   avg_ctr      n
0             0-30  0.609021  20480
1            31-90  0.117543    175
2           91-180  0.238367   9171
3             180+  3.693276    174


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

The ranking score is based on the **CTR gap**, which is calculated as the difference between the expected CTR for a page's search position bucket and its actual CTR. Only positive gaps are retained, so pages performing at or above expectations receive a score of zero. Pages are then ranked from highest to lowest score. Each page is assigned the reason code **`CTR_UNDERPERFORM`** and an action of **`rewrite_title`** if its score is positive; otherwise, **`no_action`**. The final ranked queue is saved as **`work/outputs/baseline_action_score.csv`**.


In [17]:
expected_ctr = df.groupby('position_bucket', observed=True)['ctr'].transform('mean')
df['ctr_gap'] = expected_ctr - df['ctr']
df['score'] = df['ctr_gap'].clip(lower=0)
df['reason_code'] = 'CTR_UNDERPERFORM'
df['action'] = df['score'].apply(lambda x: 'rewrite_title' if x > 0 else 'no_action')

queue = df.sort_values('score', ascending=False)

import os
os.makedirs("work/outputs", exist_ok=True)
queue.to_csv("work/outputs/baseline_action_score.csv", index=False)

print(queue[['content_id','avg_position','ctr','score','reason_code','action']].head(10))


                 content_id  avg_position  ctr     score       reason_code  \
71     content_ced59bb3a1a6           2.7  0.0  2.714303  CTR_UNDERPERFORM   
25148  content_753b597f5220           3.0  0.0  2.714303  CTR_UNDERPERFORM   
13284  content_4a6afe6d93cd           3.0  0.0  2.714303  CTR_UNDERPERFORM   
17618  content_175ea196d3e0           0.3  0.0  2.714303  CTR_UNDERPERFORM   
27056  content_5919b351bfb8           0.2  0.0  2.714303  CTR_UNDERPERFORM   
28081  content_a37ce8ddc090           2.3  0.0  2.714303  CTR_UNDERPERFORM   
8269   content_e7a9fc6d28b4           2.2  0.0  2.714303  CTR_UNDERPERFORM   
8270   content_d45265ca5145           0.7  0.0  2.714303  CTR_UNDERPERFORM   
8250   content_f550c4514619           2.7  0.0  2.714303  CTR_UNDERPERFORM   
8210   content_ad56df47b13c           2.5  0.0  2.714303  CTR_UNDERPERFORM   

              action  
71     rewrite_title  
25148  rewrite_title  
13284  rewrite_title  
17618  rewrite_title  
27056  rewrite_title  
280

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [18]:
top10 = queue.head(10)
print(top10[['content_id','avg_position','ctr','score']])


                 content_id  avg_position  ctr     score
71     content_ced59bb3a1a6           2.7  0.0  2.714303
25148  content_753b597f5220           3.0  0.0  2.714303
13284  content_4a6afe6d93cd           3.0  0.0  2.714303
17618  content_175ea196d3e0           0.3  0.0  2.714303
27056  content_5919b351bfb8           0.2  0.0  2.714303
28081  content_a37ce8ddc090           2.3  0.0  2.714303
8269   content_e7a9fc6d28b4           2.2  0.0  2.714303
8270   content_d45265ca5145           0.7  0.0  2.714303
8250   content_f550c4514619           2.7  0.0  2.714303
8210   content_ad56df47b13c           2.5  0.0  2.714303


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Rows 7 and 8 look like the weakest picks. Their CTR gap appears small and close to the bucket cutoff, making the recommendation less convincing. No future data or product-specific flags were used. The model relies only on avg_position, ctr, and days_since_last_update, all of which are available at prediction time.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.